# Agenda 

The goal of this notebook is to ensure eveything is setup for the workshop

## Setup

In [1]:
%pip install -r ../requirements.txt

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Imports

In [2]:
import dotenv
import os
import requests
import rich

In [3]:
dotenv.load_dotenv("../.env", override=True)

True

# Check Environment Variables

In [4]:
env_vars_to_check = [
    "OPENAI_API_KEY",
    "OPENAI_BASE_URL",
    "OPENAI_MODEL",
    "TAVILY_API_KEY",
    "TAVILY_BASE_URL",
    "PHOENIX_COLLECTOR_ENDPOINT",
    "PHOENIX_PROJECT_NAME",
]

def _mask(value: str, keep: int = 4) -> str:
    if value is None:
        return "<NOT SET>"
    if len(value) <= keep * 2:
        return "*" * len(value)
    return f"{value[:keep]}...{value[-keep:]}"


for name in env_vars_to_check:
    value = os.environ.get(name)
    globals()[name] = value
    if "KEY" in name and value is not None:
        print(f"{name}={_mask(value)}")
    else:
        print(f"{name}={value if value is not None else '<NOT SET>'}")


OPENAI_API_KEY=sk-o...aafb
OPENAI_BASE_URL=https://openrouter.ai/api/v1
OPENAI_MODEL=openai/gpt-oss-120b:free
TAVILY_API_KEY=tvly...GCjj
TAVILY_BASE_URL=https://api.tavily.com/search
PHOENIX_COLLECTOR_ENDPOINT=<NOT SET>
PHOENIX_PROJECT_NAME=sina-deep-research


## Check LLM (OpenAI compatible endpoint)

In [5]:
from langchain_openai import ChatOpenAI

In [6]:

llm = ChatOpenAI(
    model=OPENAI_MODEL,
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    #temperature=0.2,
    #max_tokens=512,
)

llm.invoke("what is the weather in seattle")

AIMessage(content='I’m not able to pull live weather data, so I can’t give you the current conditions in Seattle right now.  \n\nIf you need an up‑to‑date forecast, the quickest ways are:\n\n1. **Weather websites or apps** – check sites like Weather.com, AccuWeather, or the National Weather Service, or open the built‑in weather app on your phone.  \n2. **Voice assistants** – ask Siri, Google Assistant, or Alexa, e.g., “What’s the weather in Seattle?”  \n3. **Search engine** – type “Seattle weather” into Google or Bing and the current temperature, chance of rain, and a short‑term forecast will appear at the top of the results.\n\nSeattle’s climate is typically mild and wet, especially in the fall and winter, with cooler, drier summers. If you let me know the date range you’re interested in (e.g., “next weekend”), I can share typical historical averages for that time of year, but for the exact current forecast you’ll need one of the real‑time sources above.', additional_kwargs={'refusal'

## Check Tavilly (Search API)



[Tavily](https://www.tavily.com/) is a service provider that enables agents to access the web

In [7]:
def tavily_search(query, **kw):
    if TAVILY_API_KEY is None:
        r = requests.post(TAVILY_BASE_URL,
                        json={"query": query, **kw}, timeout=60)
        r.raise_for_status()
        return r.json()
    else:
        from tavily import TavilyClient
        tavily_client = TavilyClient(api_key=TAVILY_API_KEY)
        return tavily_client.search(query=query, **kw)


res = tavily_search("what is the weather in seattle")
print(res)


{'query': 'what is the weather in seattle', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'Weather in Seattle', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Seattle', 'region': 'Washington', 'country': 'United States of America', 'lat': 47.6064, 'lon': -122.3308, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1778997484, 'localtime': '2026-05-16 22:58'}, 'current': {'last_updated_epoch': 1778996700, 'last_updated': '2026-05-16 22:45', 'temp_c': 8.3, 'temp_f': 46.9, 'is_day': 0, 'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/night/116.png', 'code': 1003}, 'wind_mph': 2.2, 'wind_kph': 3.6, 'wind_degree': 207, 'wind_dir': 'SSW', 'pressure_mb': 1023.0, 'pressure_in': 30.2, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 86, 'cloud': 50, 'feelslike_c': 8.4, 'feelslike_f': 47.0, 'windchill_c': 6.3, 'windchill_f': 43.3, 'heatindex_c': 7.9, 'heatindex_f': 46.2, 'dewpoint_c': 5.4, 'dewpoint_f'

In [8]:
rich.print(res)

{
    'query': 'what is the weather in seattle',
    'follow_up_questions': None,
    'answer': None,
    'images': [],
    'results': [
        {
            'title': 'Weather in Seattle',
            'url': 'https://www.weatherapi.com/',
            'content': "{'location': {'name': 'Seattle', 'region': 'Washington', 'country': 'United States of 
America', 'lat': 47.6064, 'lon': -122.3308, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1778997484, 
'localtime': '2026-05-16 22:58'}, 'current': {'last_updated_epoch': 1778996700, 'last_updated': '2026-05-16 22:45',
'temp_c': 8.3, 'temp_f': 46.9, 'is_day': 0, 'condition': {'text': 'Partly cloudy', 'icon': 
'//cdn.weatherapi.com/weather/64x64/night/116.png', 'code': 1003}, 'wind_mph': 2.2, 'wind_kph': 3.6, 'wind_degree':
207, 'wind_dir': 'SSW', 'pressure_mb': 1023.0, 'pressure_in': 30.2, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity':
86, 'cloud': 50, 'feelslike_c': 8.4, 'feelslike_f': 47.0, 'windchill_c': 6.3, 'windchill_f': 43.3, 'heatindex_c': 
7.9, 'heatindex_f': 46.2, 'dewpoint_c': 5.4, 'dewpoint_f': 41.7, 'vis_km': 16.0, 'vis_miles': 9.0, 'uv': 0.0, 
'gust_mph': 3.3, 'gust_kph': 5.3, 'will_it_rain': 0, 'chance_of_rain': 0, 'will_it_snow': 0, 'chance_of_snow': 
0}}",
            'score': 0.9932381,
            'raw_content': None
        },
        {
            'url': 'https://www.predictwind.com/weather/united-states/washington/seattle/may',
            'title': 'Historical Weather: Seattle, United States (may 2026) - PredictWind',
            'content': '# Historical Weather: Seattle, United States (may 2026) • PredictWind. 
Weather/Browse/United States/Washington/Seattle/may. # Seattle Weather History. Historical data for United States. 
This data is based on historical weather records for Seattle, United States in may 2026. The weather in this month 
is generally cool, with breezy conditions and a dry climate. 16.4 mm Total Monthly. ### Daily Rainfall (mm). | 26 -
| 27 - | 28 - | 29 - | 30 - | 1 15° | 2 15° |. | 31 - | 1 15° | 2 15° | 3 18° | 4 18° | 5 16° | 6 15° |. Get the 
app for the world’s most accurate hi-res weather forecasts. ProductsIridium PhonesPredictWind AppOffshore 
AppIridium GO! FeaturesWeather RoutingPower RoutingDeparture PlanningCurrent ModelsGPS TrackingMapsDaily 
BriefingGraphs/TablesWeather ModelsAlertsObservationsLocal KnowledgeValidationClimate DataAIS DataAI 
PolarsMarinasGlossaryWeather. AboutTake a TourWhy PredictWindTestimonialsNewsPricingGO!/GO! Iridium 
PhonesPredictWind AppOffshore AppIridium GO! Weather RoutingPower RoutingDeparture PlanningCurrent ModelsGPS 
TrackingMapsDaily BriefingGraphs/TablesWeather ModelsAlertsObservationsLocal KnowledgeValidationClimate DataAIS 
DataAI PolarsMarinasGlossaryWeather. Take a TourWhy PredictWindTestimonialsNewsPricingGO!/GO!',
            'score': 0.8627572,
            'raw_content': None
        },
        {
            'url': 'https://www.weather25.com/north-america/usa/washington/seattle?page=month&month=May',
            'title': 'Seattle weather in May 2026 - Weather25.com',
            'content': '# Seattle weather in May 2026. ## The average weather in Seattle in May. The temperatures 
in Seattle in May are quite cold with **temperatures between 7°C and 19°C**, warm clothes are a must. You can 
expect about **3 to 8 days of rain** in Seattle during the month of May. It’s a good idea to bring along your 
umbrella so that you don’t get caught in poor weather. Our weather forecast can give you a great sense of what 
weather to expect in **Seattle in May 2026**. | 3 Light rain 16° /6° | 4 Light rain shower 16° /5° | 5 Partly 
cloudy 16° /7° | 6 Light rain shower 16° /8° | 7 Light rain shower 15° /7° | 8 Partly cloudy 18° /6° | 9 Overcast 
21° /8° |. | May | **19°** / 7° | 5 | 27 | 0 | 39 mm | Good | Seattle in May |. The weather in Seattle in May is 
good.',
            'score': 0.85421735,
            'raw_content': None
        },
        {
            'url': 'https://world-weather.info/for

In [9]:
res = tavily_search("what is the best running shoes")
rich.print(res)

{
    'query': 'what is the best running shoes',
    'follow_up_questions': None,
    'answer': None,
    'images': [],
    'results': [
        {
            'url': 'https://theruntesters.com/running-shoes/the-best-running-shoes-to-buy/',
            'title': 'The Best Running Shoes 2026 - The Run Testers',
            'content': '# The Best Running Shoes 2026. We’ve been testing the best running shoes available for over
a decade, but we still get very excited when we come across a new shoe that shines. 2025 was an outstanding year 
for running shoes, and many of the launches last year remain the best shoes you can get today, but there have also 
already been some excellent new arrivals in 2026 to consider. But with so many running shoes available, how do you 
find the ones that’ll work best for you? Check out our list of the best carbon plate running shoes**. Below that 
you’ll find more info on the best running shoes available in a wider variety of categories, including the best shoe
for beginners and best value running shoes. Some might prefer a more cushioned daily trainer, such as the Nike 
Vomero Plus or Asics Novablast 5, and you’ll find great options for that in our cushioned shoes section below.',
            'score': 0.9997241,
            'raw_content': None
        },
        {
            'url': 'https://sixminutemile.com/post/six-of-the-best-running-shoes-of-the-year/',
            'title': 'Best Running Shoes of the Year: 6 Top-Shelf Models',
            'content': '# Six of the Very Best Running Shoes of the Year. Shoes of the Year 2025. ***Here are six 
of the best running shoes of the year we wear-tested in 2025.***. Every year, several hundred new models of running
shoes hit running stores, and every one is slightly different. The good news is that, in this golden age of running
shoes, almost every single pair is made with high-quality, modern materials: lightweight, responsive midsole foams 
that offer exceptional energy return, engineered uppers that offer breathability and a secure fit, and advanced 
outsoles for traction and durability. Finding the best shoe for you is no easy task, but starting at your local 
running shop and trying on several pairs is always a very good start. **Why It’s Great:** In a word, it’s 
versatile. **Why You’ll Love It:** The Megablast is one of those shoes you put on and it almost feels effortless to
run in at any pace.',
            'score': 0.9996594,
            'raw_content': None
        },
        {
            'url': 'https://runrepeat.com/guides/best-running-shoes',
            'title': '7 Best Running Shoes in 2026 - RunRepeat',
            'content': '*   [Top](https://runrepeat.com/guides/best-running-shoes). *   [How we test running 
shoes](https://runrepeat.com/guides/best-running-shoes#how-we-test-running-shoes). *   [Best 
overall](https://runrepeat.com/guides/best-running-shoes#best-overall). *   [Best for 
tempo](https://runrepeat.com/guides/best-running-shoes#best-for-tempo). *   [Best for 
race](https://runrepeat.com/guides/best-running-shoes#best-for-race). *   [Best 
stability](https://runrepeat.com/guides/best-running-shoes#best-stability). *   
[Comparison](https://runrepeat.com/guides/best-running-shoes#comparison). trail running 
shoes](https://runrepeat.com/guides/best-running-shoes#where-to-start-road-vs-trail-running-shoes). *   [Trail 
running shoes](https://runrepeat.com/guides/best-running-shoes#trail-running-shoes). *   [Consider the distance as 
well](https://runrepeat.com/guides/best-running-shoes#consider-the-distance-as-well). *   [Foam 
softness](https://runrepeat.com/guides/best-running-shoes#foam-softness). *   [Consider breathability and 
waterproofing](https://runrepeat.com/guides/best-running-shoes#consider-breathability-and-waterproofing). *   
[Traction in running shoes](https://runrepeat.com/guides/best-running-shoes#traction-in-running-shoes). ![Image 2: 
7 Best Running Shoes in 
2026](https://cdn.runrepeat.com/storage/gallery/bu

## Check Phoenix (Observability)

Check to see if Phoenix server is up and running, otherwise start one

In [11]:
import subprocess
import time
import requests as _requests

PHOENIX_PORT = 6006
PHOENIX_STARTUP_TIMEOUT = 10  # seconds to wait for Phoenix to become ready

def _is_phoenix_ready(port: int) -> bool:
    """Return True if a Phoenix server is already up and responding on the given port."""
    try:
        r = _requests.get(f"http://localhost:{port}/healthz", timeout=2)
        return r.status_code == 200
    except Exception:
        return False

if PHOENIX_COLLECTOR_ENDPOINT:
    print(f"Phoenix endpoint already configured: {PHOENIX_COLLECTOR_ENDPOINT}")
elif _is_phoenix_ready(PHOENIX_PORT):
    print(f"Phoenix already running and healthy on port {PHOENIX_PORT}, reusing it.")
    PHOENIX_COLLECTOR_ENDPOINT = f"http://localhost:{PHOENIX_PORT}"
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = PHOENIX_COLLECTOR_ENDPOINT
else:
    print("Starting local Phoenix server...")
    subprocess.Popen(
        ["python", "-m", "phoenix.server.main", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    for _ in range(PHOENIX_STARTUP_TIMEOUT):
        if _is_phoenix_ready(PHOENIX_PORT):
            break
        time.sleep(1)
    else:
        raise RuntimeError(f"Phoenix did not become ready within {PHOENIX_STARTUP_TIMEOUT} seconds.")
    PHOENIX_COLLECTOR_ENDPOINT = f"http://localhost:{PHOENIX_PORT}"
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = PHOENIX_COLLECTOR_ENDPOINT
    print(f"Phoenix UI: {PHOENIX_COLLECTOR_ENDPOINT}")
    print(f"PHOENIX_COLLECTOR_ENDPOINT set to: {PHOENIX_COLLECTOR_ENDPOINT}")


Phoenix already running and healthy on port 6006, reusing it.


In [12]:
from phoenix.otel import register
from opentelemetry import trace

from openinference.instrumentation.langchain import LangChainInstrumentor
if PHOENIX_COLLECTOR_ENDPOINT:
    # configure the Phoenix tracer
    tracer_provider = register(
        project_name=PHOENIX_PROJECT_NAME, 
        auto_instrument=False 
    )
else:
    tracer_provider = trace.NoOpTracerProvider()

LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
tracer = trace.get_tracer(__name__)


/home/vscode/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/vscode/.local/lib/python3.13/site-packages/phoenix/server/agents/pydantic_ai/openinference_agent_wrapper.py:28: PydanticAIDeprecationWarning: `pydantic_ai.messages.BuiltinToolCallPart` is deprecated, use `pydantic_ai.messages.NativeToolCallPart` instead.
  from pydantic_ai.messages import (
/home/vscode/.local/lib/python3.13/site-packages/phoenix/server/agents/pydantic_ai/openinference_agent_wrapper.py:28: PydanticAIDeprecationWarning: `pydantic_ai.messages.BuiltinToolReturnPart` is deprecated, use `pydantic_ai.messages.NativeToolReturnPart` instead.
  from pydantic_ai.messages import (


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: sina-deep-research
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



# LangChain

In [13]:
from langchain.agents import create_agent

In [14]:

from langchain_openai import ChatOpenAI

def get_weather(city: str) -> str:  
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

llm = ChatOpenAI(
    model=OPENAI_MODEL,
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY
)

agent = create_agent(
    llm,
    tools=[get_weather],  
    system_prompt="You are a helpful assistant"  
)

res = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in seattle"}]}
)
